In [1]:
# import contextlib

# with contextlib.suppress(ImportError):
#     from google.colab import drive  # type: ignore

#     drive.mount("/content/drive")
#     !cp /content/drive/MyDrive/BuildGPT/Part4/name.txt /content/

try: 
    from google.colab import drive  # type: ignore
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    !cp /content/drive/MyDrive/BuildGPT/Part4/name.txt /content/ 

In [3]:
import torch
import torch.nn.functional as F
# import matplotlib.pyplot as plt  # for making figures
import random

In [4]:
with open("name.txt", "r") as file:
    words = file.read().splitlines()
print(len(words))
print(max(len(w) for w in words))  # 生成器表达式
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [5]:
# join 组合为字符串, set-字符级去重,
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [6]:
block_size = 3


def build_dataset(words):
    # context_code 是 character 的索引
    # X-context_code, Y-prediction_code(next)
    X, Y = [], []
    for word in words:
        context_code = [0] * block_size  # type: list[int]
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context_code)  # type: list[list[int]]
            Y.append(ix)  # type: list[int]
            context_code = context_code[1:] + [ix]
    # print((X[0]))
    # print((Y[0]))
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y


# build_dataset(words)


In [7]:
random.seed(42)
random.shuffle(words)
len_words = len(words)
n1 = int(0.8 * len_words)
n2 = int(0.9 * len_words)

X_train, Y_train = build_dataset(words[:n1])
X_dev, Y_dev = build_dataset(words[n1:n2])
X_ver, Y_ver = build_dataset(words[n2:])

print(X_train.shape)
print(Y_train.shape)
print(X_dev.shape)
print(Y_dev.shape)
print(X_ver.shape)
print(Y_ver.shape)

torch.Size([182625, 3])
torch.Size([182625])
torch.Size([22655, 3])
torch.Size([22655])
torch.Size([22866, 3])
torch.Size([22866])


In [8]:
n_embd = 10  # 字符嵌入向量的维度(n_embd 个特征描述一个字符)
n_hidden = 64  # MLP 所包括的神经网络的数量

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)  # 查找表

# layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (
    (5 / 3) / (n_embd * block_size) ** 0.5
)
b1 = torch.randn(n_hidden, generator=g) * 0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.
bn_gain = torch.randn((1, n_hidden)) * 0.1 + 1.0  # 行自动扩展
bn_bias = torch.randn((1, n_hidden)) * 0.1  # 行自动扩展

# layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1


parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
print(sum(p.nelement() for p in parameters))  # number of parameters in total


4137


假设公式本该是 $y = a \times b$

误写成了 $y = a + b$

如果 $a=0, b=0$：正确公式的结果是 $0$。错误公式的结果也是 $0$。

结果：你的输出是对的，甚至梯度计算在这一步可能也是 0。你看着损失函数在下降，但其实底层的数学逻辑是完全错误的。

Karpathy 的意思是： 如果我用随机数（甚至是奇奇怪怪的非标准随机数）初始化，一旦我的反向传播实现有误，计算出来的梯度会立刻变得非常荒谬（比如巨大无比或变成 NaN），我能瞬间发现代码写错了。

具体的数学陷阱
A. 对称性破缺 (Symmetry Breaking)如果你把 W1 和 W2 全部初始化为 0：所有的神经元在第一轮计算时，输出完全一样。反向传播时，它们的梯度也完全一样。结果：无论你有多少个神经元，它们最终都会变成一模一样的分身。这让多神经元的网络退化成了单神经元

...

In [9]:
batch_size = 32
ix = torch.randint(
    0, X_train.shape[0], (batch_size,), generator=g
)  # 随机数, 范围[0,  X_train 的行), ix 是一维张量
X_train_batch, Y_train_batch = (
    X_train[ix],
    Y_train[ix],
)  # 从 X_train 中取 batch_size 行, 行号为 ix 中的随机数

In [10]:
############# forward begin ################
for p in parameters:
    p.requires_grad = True

emb_batch = C[X_train_batch]  # C 嵌入到 X_train_batch
embcat_batch = emb_batch.view(
    emb_batch.shape[0], -1
)  # 逻辑上视为二维, 行确定, 列自动推断
h_preact_batch = embcat_batch @ W1 + b1

""" batch normalization begin"""

# 行压缩, 虽然结果是一行, 但保留维度, shape 为(1, h_preact_bn[1]), 而不是(,h_preact_bn)
# 为什么压缩的是行? 因为这样求出的平均值是列平均值, 而列平均值表示单个神经元所有参数的平均值
bn_meani = h_preact_batch.sum(0, keepdim=True) * (1 / batch_size)
# bn_meani 会 broadcast(自动扩展行)
# bn_diff 的均值是 0
bn_diff = h_preact_batch - bn_meani
bn_diff2 = bn_diff**2
bn_var = bn_diff2.sum(0, keepdim=True) * (
    1 / (batch_size - 1)
)  # 修改：除以 (n-1) 而不是 n（Bessel修正), 详细解释见下
bn_var_inv = (bn_var + 1e-5) ** (-0.5)
bn_raw = bn_diff * bn_var_inv  # 均值 0, 方差 1, 就是经过处理的 h_preact_batch
bn_hpreact = bn_gain * bn_raw + bn_bias

"""batch normalization end"""

h = torch.tanh(bn_hpreact)  # 修改：用 bn_hpreact 而不是 h_preact_batch
logits = h @ W2 + b2  # 认为是对数空间, 有正有负, 详细解释见下

"""cross entropy loss begin"""

"""softmax begin"""

# 一行表示该 batch 中的一个样本(block_size 个字母), 而列的值表示经过最终综合评估, 针对该样本预测的下一个的字母的概率(权重)
logits_max = logits.max(1, keepdim=True).values  # shape: (32, 1)

# 保持值最小(负数),避免e^x 浮点数溢出, 但相对权重不变
# 这里 logits_normal 并非归一化, 而是平移, 稳定规范化
logits_normal = logits - logits_max
counts = logits_normal.exp()  # 这里直接使用了概率的概念, 意思是次数,频次
counts_sum = counts.sum(1, keepdim=True) # 频次和
counts_sum_inv = counts_sum ** (-1)
probs = counts * counts_sum_inv  # 频次 / 总和 = 概率

"""soft max end"""


""" Negative Log Likelihood, NLL begin"""
# 详细解释见下面
probs_log = probs.log()
temp = probs_log[range(batch_size), Y_train_batch]  # for test only
loss = -probs_log[range(batch_size), Y_train_batch].mean()

"""Negative Log Likelihood, NLL end"""

"""cross entropy loss end"""
############# forward end ################

'cross entropy loss end'

In [11]:
############# backward begin ################

# 为了支持超大 Batch, pytorch 支持分多个 small batch 训练模型, 因而 pytorch 设计为当 backward 时累加 grad, 而不是覆盖
# 这里为了获取本次准确的 grad, 因而清除
for p in parameters:
    p.grad = None  # 清除原始梯度

for t in [
    probs_log,
    probs,
    counts,
    counts_sum,
    counts_sum_inv,  # afaik there is no cleaner way
    logits_normal,
    logits_max,
    logits,
    h,
    bn_hpreact,
    bn_raw,
    bn_var_inv,
    bn_var,
    bn_diff2,
    bn_diff,
    bn_meani,
    h_preact_batch,
    embcat_batch,
    emb_batch,
]:
    t.retain_grad()
loss.backward()
loss
############# backward end ################

tensor(3.3293, grad_fn=<NegBackward0>)

*嵌入的本质?*

// `emb_batch = C[X_train_batch]`

// `X_train_batch` 中存储的是字母的索引

// `X_train_batch: [B][T], C: [V][D], emb_batch: [B][T][D]`

// 遍历 `X_train_batch`
```c++
for (int b = 0; b < B; ++b) {
    for (int t = 0; t < T; ++t) {
        int vocab_idx = X_train_batch[b][t]; // 获取当前的单词索引
        // 将对应的嵌入向量拷贝到输出张量的正确位置
        // C[vocab_idx], 第几行就是对应的字母
        // emb_batch 本质是使用了 X_train_batch 的形状
        // - 区别在于 X_train_batch 的元素是一维的值, 是一个索引, 而 emb_batch 的元素是一个 shape 为 [D,1] 的 tensor
        // - 其实就是将 C 中的特定行 copy 到了 X_train_batch 的对应位置, 将 C 嵌入了 X_train_batch
        memcpy(&emb_batch[b][t][0], &C[vocab_idx][0], D * sizeof(float));
    }
}
```
---

*`bn_var` 为什么要除以 $n-1$?*

(Bessel's Correction)这是统计学中的贝塞尔修正 (Bessel's Correction)。

样本方差 (Sample Variance)：当你只有一部分数据（一个 Batch）时，你计算出的方差往往会比总体（整个数据集）的真实方差偏小。

无偏估计 (Unbiased Estimator)：为了纠正这个偏差，数学上证明了除以 $n-1$ 比除以 $n$ 能更准确地估计总体的真实方差。

在深度学习的实际生产环境（如 torch.nn.BatchNorm1d）中，通常默认除以 $n$。

Karpathy 在这里手写 $n-1$ 是为了展示最严谨的统计学定义

___

*Logit  的含义是什么?*

在数学和统计学中，Logit 函数 是概率的“对数几率”。

但在深度学习的工程实践中，它的含义非常具体：指神经网络最后一层线性层（Linear Layer）的原始输出，即在进入 Softmax（归一化为概率）之前的那些数值。

特点：范围是 $(-\infty, +\infty)$。它代表了模型对各个类别的“未归一化的信心分”。

___

*为什么是 `batch_size-1`?*
```python
counts_sum_inv = counts_sum **(-1)
```

`x ** -1 vs 1.0 / x`：指令集的差异在 GPU（尤其是 NVIDIA GPU）底层，它们映射的硬件指令可能完全不同。

1.0 / x (Standard Division)这是标准的除法指令。在现代 GPU 上，为了追求速度，它通常使用 Newton-Raphson 迭代 或专门的硬件电路来实现，遵循 IEEE 754 标准，精度较高但速度相对慢一点。

x ** -1 (Power function)在底层，pow(x, -1) 往往被解析为：调用 log2(x)乘以 -1调用 exp2(...)或者是调用专门的 RCP (Reciprocal) 快速近似指令。

关键点：RSQRT 与 RCPGPU 为了图形渲染，内置了极快的近似指令（如 RCP 求倒数，RSQRT 求平方根倒数）。

1.0 / x 可能会走高精度的除法单元。

x ** -1 可能会触发编译器的优化，走快速近似指令。

虽然误差可能只有 $2^{-23}$ 那么大，但在反向传播中，梯度会经过成百上千次的连乘，这个微小的 bit 差异会被迅速放大。

为什么要追求 "Bit Exact" (位精确)？Karpathy 之所以死磕这一点，是因为他在做 单元测试（Unit Testing）。

---
*为什么计算 loss 要取对数?*

我们的目的是计算所有预测都正确的概率, 所以依次从 prob 中找到所有正确答案的概率, 然后相乘, 正确的答案(字母的索引)保存在 Y_train_batch 中

假设 Y_train_batch[4] 是 7, 则说明该batch 的第 5 个样本期望预测的字母 g

然后看一下 prob[4][7], 假设为 0.225, 说明预测正确的概率是 0.225

实际我们应该遍历 Y_train_batch, 在 prob 中找到所有的正确答案, 那么预测全部正确的概率为

$$
\begin{align}
P&= p_0 * p_1 * ... * p_{batch\_size - 1} \hspace{100cm}\\
&\small \text{但是由于概率 < 1, 导致 P 极小, 浮点数下溢, 因而对两边同时取对数} \\
ln(P) &= ln(p_0 * p_1 * ... p_{batch\_size - 1}) \\
&=ln(p_0) + ln(p_1) + ... + * ln(p_{batch\_size - 1})) \\
\end{align}
$$
这样, 实际计算的就是概率的对数的和, 而不是概率的乘积

但是, 这样还是有问题, 由于概率 $p_i < 1$, 因而最终的 $ln(P) < 0$, 因而加一个**负号**, 将 `P` 调整为正值, 现在的关系是:

预测准确率越高, $ln(P)$ 越大(在负坐标轴部分越靠近原点), $-ln(P)$越小, 所以令 $loss = -ln(P)$

最后一个问题, 由于`loss`是求和计算得到的, 如果`batch_size`变大或变小, 由于数量剧变可能导致`loss`剧变, 因而通过求平均避免这种问题



In [12]:
def cmp(p_name: str, input_grad: torch.Tensor, other_grad: torch.Tensor):
    ex = torch.all(input_grad == other_grad).item()  # 完全相等
    app = torch.allclose(input_grad, other_grad)  # 近似(综合评估绝对差和相对差)
    max_diff = (input_grad - other_grad).abs().max().item()
    print(
        f"{p_name:15s}: exact|{str(ex):5s} || approximate|{str(app):5s} || max_diff|{max_diff}"
    )


In [13]:
print(f"Y_train_batch shape:{Y_train_batch.shape}")
print(f"probs shape:{probs.shape}")
print(f"temp shape:{temp.shape}")
print(f"loss shape:{loss.shape}")

print(f"logits shape: {logits.shape}")
print(f"logits_normal shape: {logits_normal.shape}")
print(f"counts shape:{counts.shape}")
print(f"counts_sum_inv shape:{counts_sum_inv.shape}")

print(f"h shape: {h.shape}")
print(f"W2 shape: {W2.shape}")
print(f"b2 shape: {b2.shape}")

print(f"bn_preact shape: {bn_hpreact.shape}")

print(f"bn_raw shape: {bn_raw.shape}")
print(f"bn_gain shape: {bn_gain.shape}")
print(f"bn_bias shape: {bn_bias.shape}")

print(f"bn_diff shape: {bn_diff.shape}")
print(f"bn_var_inv shape: {bn_var_inv.shape}")

print(f"bn_diff shape: {bn_diff.shape}")
print(f"bn_var_inv shape: {bn_var_inv.shape}")
print(f"bn_var shape: {bn_var.shape}")
print(f"bn_diff2 shape: {bn_diff2.shape}")

print(f"h_preact_batch shape: {h_preact_batch.shape}")
print(f"bn_meani shape: {bn_meani.shape}")

print(f"embcat_batch shape: {embcat_batch.shape}")
print(f"W1 shape: {W1.shape}")
print(f"b1 shape: {b1.shape}")

print(f"emb_batch shape: {emb_batch.shape}")
print(f"C shape: {C.shape}")
print(f"X_train_batch shape: {X_train_batch.shape}")
# print(f"h_preact_batch shape: {h_preact_batch.shape}")

Y_train_batch shape:torch.Size([32])
probs shape:torch.Size([32, 27])
temp shape:torch.Size([32])
loss shape:torch.Size([])
logits shape: torch.Size([32, 27])
logits_normal shape: torch.Size([32, 27])
counts shape:torch.Size([32, 27])
counts_sum_inv shape:torch.Size([32, 1])
h shape: torch.Size([32, 64])
W2 shape: torch.Size([64, 27])
b2 shape: torch.Size([27])
bn_preact shape: torch.Size([32, 64])
bn_raw shape: torch.Size([32, 64])
bn_gain shape: torch.Size([1, 64])
bn_bias shape: torch.Size([1, 64])
bn_diff shape: torch.Size([32, 64])
bn_var_inv shape: torch.Size([1, 64])
bn_diff shape: torch.Size([32, 64])
bn_var_inv shape: torch.Size([1, 64])
bn_var shape: torch.Size([1, 64])
bn_diff2 shape: torch.Size([32, 64])
h_preact_batch shape: torch.Size([32, 64])
bn_meani shape: torch.Size([1, 64])
embcat_batch shape: torch.Size([32, 30])
W1 shape: torch.Size([30, 64])
b1 shape: torch.Size([64])
emb_batch shape: torch.Size([32, 3, 10])
C shape: torch.Size([27, 10])
X_train_batch shape: torc

In [22]:
# d(Loss) / d(probs_log)
dprobs_log = torch.zeros_like(probs_log)
dprobs_log[range(batch_size), Y_train_batch] = -1 / batch_size
if probs_log.grad is not None:
    cmp("dprobs_log", dprobs_log, probs_log.grad)

# d(Loss) / d(probs)
# d(log(x))/d(x) = 1/x, 即 d(probs_log) / d(probs) = 1 / probs
dprobs = (1 / probs) * dprobs_log  # 链式法则
if probs.grad is not None:
    cmp("dprobs", dprobs, probs.grad)

# d(Loss) / d(counts_sum_inv)
dcounts_sum_inv = (dprobs * counts).sum(1, keepdim=True)
if counts_sum_inv.grad is not None:
    cmp("dcounts_sum_inv", dcounts_sum_inv, counts_sum_inv.grad)

# d(Loss) / d(counts)
# d(probs) / d(counts)
dcounts_1 = dprobs * counts_sum_inv


# d(Loss) / d(counts_sum)
# d(1/x) / d(x) = -(1/(x^2)), 即 d(counts_sum_inv) / d(counts_sum) = -(1 / (counts_sum^2))
dcounts_sum = dcounts_sum_inv * (-(counts_sum ** (-2)))
if counts_sum.grad is not None:
    cmp("dcounts_sum", dcounts_sum, counts_sum.grad)

# d(counts_sum) / d(counts)
dcounts_2 = dcounts_sum * torch.ones_like(counts)  # 解释见下方

dcounts = dcounts_1 + dcounts_2

if counts.grad is not None:
    cmp("dcounts", dcounts, counts.grad)

# # d(Loss) / d(logits_normal)
# d(e^x) / d(x) = e^x, 即 d(logits_normal.exp()) / dlogits_normal() = logits_normal.exp()
dlogits_normal = dcounts * logits_normal.exp()
if logits_normal.grad is not None:
    cmp("dlogits_normal", dlogits_normal, logits_normal.grad)

# d(logits_normal) / d(logits)
dlogits_1 = dlogits_normal * torch.ones_like(logits)

# d(Loss) / d(logits_max)
dlogits_max = (dlogits_normal * (-torch.ones_like(logits))).sum(
    1, keepdim=True
)  # 实际上 dlogits_max 为0 ,分析见下
if logits_max.grad is not None:
    cmp("dlogits_max", dlogits_max, logits_max.grad)

# d(logits_nmax) / d(logits)
# max(1), 压缩列
# indices, 所有max的索引组成的tensor
# logits.shape[1], 共多少列
dlogits_2 = dlogits_max * F.one_hot(
    logits.max(1).indices, logits.shape[1]
)  # 详细解释见下

# d(Loss) / d(logits)
dlogits = dlogits_1 + dlogits_2
if logits.grad is not None:
    cmp("dlogits", dlogits, logits.grad)

# logits = h @ W2 + b2, 分析见下
db2 = (dlogits * torch.ones_like(logits)).sum(dim=0)
if b2.grad is not None:
    cmp("db2", db2, b2.grad)
dh = dlogits @ W2.transpose(0, 1)
if h.grad is not None:
    cmp("dh", dh, h.grad)

dW2 = h.transpose(0, 1) @ dlogits
if W2.grad is not None:
    cmp("dW2", dW2, W2.grad)


dbn_hpreact = dh * (1 - h**2)
if bn_hpreact.grad is not None:
    cmp("dbn_hpreact", dbn_hpreact, bn_hpreact.grad)

dbn_bias = (dbn_hpreact * torch.ones_like(dbn_hpreact)).sum(dim=0)
if bn_bias.grad is not None:
    cmp("dbn_bias", dbn_bias, bn_bias.grad)

dbn_gain = (dbn_hpreact * bn_raw).sum(dim=0)
if bn_gain.grad is not None:
    cmp("dbn_gain", dbn_gain, bn_gain.grad)

dbn_raw = dbn_hpreact * bn_gain  # dbn_hpreact * bn_gain.expand(dbn_hpreact.shape)
if bn_raw.grad is not None:
    cmp("dbn_raw", dbn_raw, bn_raw.grad)

dbn_var_inv = (dbn_raw * bn_diff).sum(dim=0)
if bn_var_inv.grad is not None:
    cmp("dbn_var_inv", dbn_var_inv, bn_var_inv.grad)

dbn_diff_1 = dbn_raw * bn_var_inv

# (bn_var + 1e-5) ** (-3/2) = bn_var_inv ** 3
# dbn_var = dbn_var_inv * ((-1/2) * bn_var_inv ** 3) # 这样会有微小误差
dbn_var = dbn_var_inv * ((-1 / 2) * (bn_var + 1e-5) ** (-3 / 2))
if bn_var.grad is not None:
    cmp("dbn_var", dbn_var, bn_var.grad)

dbn_diff2 = dbn_var.expand_as(bn_diff2) * (1 / (batch_size - 1))
if bn_diff2.grad is not None:
    cmp("dbn_diff2", dbn_diff2, bn_diff2.grad)

dbn_diff_2 = dbn_diff2 * 2 * bn_diff

dbn_diff = dbn_diff_1 + dbn_diff_2
if bn_diff.grad is not None:
    cmp("dbn_diff", dbn_diff, bn_diff.grad)

dh_preact_batch_1 = dbn_diff

dbn_meani = -dbn_diff.sum(
    dim=0
)  # (dbn_diff * torch.ones_like(dbn_diff) * -1).sum(dim = 0)
if bn_meani.grad is not None:
    cmp("dbn_meani", dbn_meani, bn_meani.grad)

dh_preact_batch_2 = dbn_meani.expand_as(h_preact_batch) * (1 / batch_size)

dh_preact_batch = dh_preact_batch_1 + dh_preact_batch_2

if h_preact_batch.grad is not None:
    cmp("dh_preact_batch", dh_preact_batch, h_preact_batch.grad)

db1 = dh_preact_batch.sum(dim=0)
if b1.grad is not None:
    cmp("db1", db1, b1.grad)

dW1 = embcat_batch.transpose(0, 1) @ dh_preact_batch
if W1.grad is not None:
    cmp("dW1", dW1, W1.grad)

dembcat_batch = dh_preact_batch @ W1.transpose(0, 1)
if embcat_batch.grad is not None:
    cmp("dembcat_batch", dembcat_batch, embcat_batch.grad)

demb_batch = dembcat_batch.view(emb_batch.shape)
if emb_batch.grad is not None:
    cmp("demb_batch", demb_batch, emb_batch.grad)

# 这里本质没有发生任何产生变化的计算, 仅仅是拷贝
dC = torch.zeros_like(C)
for i in range(X_train_batch.shape[0]): # 行
    for j in range(X_train_batch.shape[1]): # 列
        char_index = X_train_batch[i,j] # 同一个字母可能多次出现, 同一个 char_index 也会多次出现
        dC[char_index] += demb_batch[i,j] # demb_batch 的一个元素的数量和 dC 一行的数量是相等的, 详见前面的 "嵌入的本质"
if C.grad is not None:
    cmp("dC", dC, C.grad)        
# h_preact_batch = embcat_batch @ W1 + b1

dprobs_log     : exact|True  || approximate|True  || max_diff|0.0
dprobs         : exact|True  || approximate|True  || max_diff|0.0
dcounts_sum_inv: exact|True  || approximate|True  || max_diff|0.0
dcounts_sum    : exact|True  || approximate|True  || max_diff|0.0
dcounts        : exact|True  || approximate|True  || max_diff|0.0
dlogits_normal : exact|True  || approximate|True  || max_diff|0.0
dlogits_max    : exact|True  || approximate|True  || max_diff|0.0
dlogits        : exact|True  || approximate|True  || max_diff|0.0
db2            : exact|True  || approximate|True  || max_diff|0.0
dh             : exact|True  || approximate|True  || max_diff|0.0
dW2            : exact|True  || approximate|True  || max_diff|0.0
dbn_hpreact    : exact|True  || approximate|True  || max_diff|0.0
dbn_bias       : exact|True  || approximate|True  || max_diff|0.0
dbn_gain       : exact|True  || approximate|True  || max_diff|0.0
dbn_raw        : exact|True  || approximate|True  || max_diff|0.0
dbn_var_in

1. *为什么 `probs_log_grad` 要初始化为 0?*

`probs_log` 中的所有参数, 只有正确答案的参数是与 Loss 相关的, 其 grad != 0, 其它错误答案的参数与 Loss 无关, 因而其 grad == 0

*为什么`probs_log_grad`正确答案的`grad`要赋值为 -1/n?*

$$
\begin{align}
n &= batch\_size \\
\text(这里的 p 均指 p_{log}) \\
Loss &= -(p1 + p2 + ...) / n \hspace{100cm} \\
&=(-1/n)p1 + (-1/n)p2 +...
\end{align}
$$

显然 Loss 对 p1, p2 的偏导都是 -1/n

*`dprobs = (1 / probs) * dprobs_log` 的直观理解*

`1 / probs` 直观上可以理解为一个 `grad` 放大器, 或者说错误放大器

$probs < 1, \dfrac {1}{prob} > 1 $, 假如对正确答案预测的 $prob = 0.01$, 那么 $\dfrac {1}{prob} = 100 $, 这意味着`grad` 将被放大 100 倍, 模型将更容易感知到错误

---
2. *`dcounts_sum_inv`*

`probs = counts * counts_sum_inv` 发生了 broadcast

$$
\begin{align}
& probs_{i0} = counts_{i0} * counts\_sum\_inv_{i0} \hspace{100cm} \\
& probs_{i1} = counts_{i1} * counts\_sum\_inv_{i0} \hspace{100cm} \\
& probs_{i2} = counts_{i2} * counts\_sum\_inv_{i0} \hspace{100cm} \\
& ... \\
& 令 V = counts\_sum\_inv \\
& \frac{\partial L}{\partial V_i} = \frac{\partial L}{\partial P_{i,1}} \cdot \frac{\partial P_{i,1}}{\partial V_i} + \frac{\partial L}{\partial P_{i,2}} \cdot \frac{\partial P_{i,2}}{\partial V_i} + \dots + \frac{\partial L}{\partial P_{i,27}} \cdot \frac{\partial P_{i,27}}{\partial V_i} \\
& \text(从 1 可以看出来, 多个p 是通过求和影响 Loss 的)\\
& probs = counts * counts\_sum\_inv, 显然这里的 counts 就是导数

\end{align}
$$

沿着一条线走（串联）：梯度相乘。（比如从 $L$ 到 $P$，再从 $P$ 到 $V$）。代码里的 dprobs * counts 就是这一步。

多条线汇聚到一个点（并联）：梯度相加。（比如 $V$ 同时影响了 27 个 $P$，这 27 条线的反馈要汇聚给 $V$）。代码里的 .sum() 就是这一步。

___

3. *`dcounts_2`*
counts_sum_i = counts_i0 + counts_i1 + counts_i2 + ...

所以每个 counts_ix 的偏导数都是1

4. *`logits_max`*

改变 logits_max 不会影响概率分布, 进而不会影响 Loss, 因而  d(Loss) / d(logits_max) = 0

5. *`dlogits_2`*

为什么是one_hot? 以一行为例, 假设 $x_4$是 max

$x_1, x_2, x_3, x_4(max), x_5, x_6$

那么 max(x) 显然与处 x4 以外的其它 x 无关, 无论它们如何变化(只要不超过x4, 超过就推翻假设了), 而max(x) = x4, 所以

$$
d(max) / d(x) =
\begin{cases}
0, &\text{if x is not max} \\
1, & \text{if x is max}
\end{cases}
$$

6. *logits = h @ W2 + b2*

```python
h shape: torch.Size([32, 64])
W2 shape: torch.Size([64, 27])
b2 shape: torch.Size([27])
logits shape: torch.Size([32, 27])
```
**`d(b2)`**

如果`b2 shape`为 [32, 27], 那 $\dfrac {d(logits)} {d(b2)}$ 应为`torch.ones_like(logits)`, $\dfrac {d(Loss)} {d(b2)}$ 为`dlogits * torch.ones_like(logits)`

但是`b2 shape`为 [27], `b2`会 broadcast, 因此, $\dfrac {d(Loss)} {d(b2)}$ 为`(dlogits * torch.ones_like(logits)).sum(dim=0)`

**`d(h), d(W2)`**

不考虑b2, b2 对 h 和 W2 的偏导没有影响

$$
\begin{align}
  logits_{00}  & = h_{00}*W2_{00} + h_{01}*W2_{10} + h_{02}*W2_{20} + ... + h_{0,64}*W2_{64,0} \\
  logits_{01}  & = h_{00}*W2_{01} + h_{01}*W2_{11} + h_{02}*W2_{21} + ... + h_{0,64}*W2_{64,1} \\
  ...\\
  logits_{027}  & = h_{00}*W2_{0,27} + h_{01}*W2_{1,27} + h_{02}*W2_{2,27} + ... + h_{0,64}*W2_{64,27} \\
\end{align}
$$

- 就 h 的第 0 行而言

  - 各个元素的局部偏导$\dfrac{d(logits)}{d(h)}$应为

$$
  \begin{align}
  &dh_{00} = W2_{00} + W2_{01} + ... + W2_{0,27} \\
  &dh_{01} = W2_{10} + W2_{11} + ... + W2_{1,27} \\
  &... \\
  &dh_{0,64} = W2_{64,0} + W2_{64, 1} + ... + W2_{64,27} \\
  \end{align}
$$

  - 各个元素的全局偏导 $\dfrac{d(Loss)}{d(h)} $应为
$$
  \begin{align}
  & W2_{00}, W2_{10}, ..., W2_{64, 0}  的因变量都是 logits_{00}\\
  & W2_{01}, W2_{11}, ..., W2_{64, 1}  的因变量都是 logits_{01}\\
  & ... \\
  & W2_{0,27}, W2_{1,27}, ..., W2_{64, 27}  的因变量都是 logits_{0,27}\\
  & 所以\\
  & dh_{00} = dlogits_{00} * W2_{00} + dlogits_{01} * W2_{01} + ... + dlogits_{0,27} * W2_{0,27} \\
  & dh_{01} = dlogits_{00} * W2_{10} + dlogits_{01} * W2_{11} + ... + dlogits_{0,27} * W2_{1,27} \\
  & ... \\
  & dh_{0,64} = dlogits_{00} * W2_{64,0} + dlogits_{01} * W2_{64, 1} + ... + dlogits_{0,27} * W2_{64,27} \\
  \end{align}
$$

- 其它行同理, 所以

  ```python
  dh = dlogits @ W2.transpose

  反之

  dW2 = h.transpose @ dlogits
  ```

In [ ]:
################# cross_entropy #########################
# softmax + NLL
loss_fast = F.cross_entropy(logits, Y_train_batch)
print(f"loss: {loss_fast.item()}, diff: {(loss_fast - loss).item()}")


In [ ]:
################# backward of cross_entropy #########################
# 快速计算 dlogits
dlogits_fast = torch.clone(probs) # 对应 pi, 区别在于 i 是一行, 这是全部

dlogits_fast[range(batch_size), Y_train_batch] -= 1.0 # 仅取出 pt, 然后 - 1
# pi - 0, 不需要做任何操作

dlogits_fast /= batch_size

if logits.grad is not None:
    cmp("dlogits_fast ", dlogits_fast, logits.grad)

dlogits_fast   : exact|False || approximate|True  || max_diff|4.889443516731262e-09


`i`, 字符表中字母的下标 [0,26]

模型预测下一个字符是第`i`个字符额概率 `p_i`

`t`, 实际正确答案的字符的下标

以 `logits - max(logits)` 中的某一个样本(某一行)为例, 设为`z`

**我们的目标是 Loss 对 计算正确答案 z_t 的导数, 即 $\dfrac{\partial{ L}}{\partial{z_t}}$**

$$
\begin{array}{l}
p_i = \dfrac{e^{z_i}}{\sum\limits_{j=0}^{n=26}{e^{z_j}}} \hspace{100cm}\\
 \\
L_i = -ln \left(\dfrac{e^{z_i}}{\sum\limits_{j=0}^{n=26}{e^{z_j}}}\right) \hspace{100cm}\\
L_i    = -\left(ln(e^{z_i}) - ln\left(\sum\limits_{j=0}^{n=26}{e^{z_j}}\right)\right)\hspace{100cm}\\
L_i   = ln\left(\sum\limits_{j=0}^{n=26}{e^{z_j}}\right) - z_i \hspace{100cm}\\

\\
\because \small \text ln(x) 的导数是 1/x, e^x 的导数是 e^x\\
\\

\small{\text {如果 \: i = t, 即预测字符与实际字符匹配}}  \\
\\
\therefore \dfrac{\partial{ L_t}}{\partial{ z_t}} = \left(\dfrac{e^{z_j}}{\sum\limits_{j=0}^{n=26}{e^{z_j}}}\right) - 1  \\
\\
\small{\text {如果}} \: i \neq t \\
\\

\therefore \dfrac{\partial{ L_t}}{\partial{ z_i}} = \left(\dfrac{e^{z_j}}{\sum\limits_{j=0}^{n=26}{e^{z_j}}}\right) - 0  \\

\because p_i = \dfrac{e^{z_i}}{\sum\limits_{j=0}^{n=26}{e^{z_j}}} \hspace{100cm} \\

\\
\therefore 
L = 
\begin{cases}
{p_i - 1}, &{i=t} (L < 0)\\
p_i,  &{i \neq t} (L > 0)
\end{cases}

\\
\\
\small{\text{令}}\\
y_i = 
\begin{cases}
1, &{i=t} \\
0,  &{i \neq t}
\end{cases}

\\
\small{\text{则}}\\
L = p_i - y_i
\end{array}
$$

---

`p.data += -p.grad * learning_rate`

p 是参数

micrograd 中提到梯度下降的公式是

p.data += -p.grad * learning_rate

就是说 grad > 0 会导致 p.data 下降, 反之导致 p.data 上升

换为 前面推导时用到的`z`
```python
# dlogits_fast 中正确字母对应的值 < 0, 而错误答案位置的值 > 0
# 所以训练过程会不断拉高正确答案的概率, 而降低错误答案的概率
z += -dlogits_fast * learning_rate
```

In [ ]:
############### batch normalization #######################
bn_hpreact_fast = (
    bn_gain
    * (h_preact_batch - h_preact_batch.mean(0, keepdim=True))
    / torch.sqrt(h_preact_batch.var(0, keepdim=True, correction=True) + 1e-5)
    + bn_bias
)

print("max diff:", (bn_hpreact_fast - bn_hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


```python
# 行压缩, 虽然结果是一行, 但保留维度, shape 为(1, h_preact_bn[1]), 而不是(,h_preact_bn)
# 为什么压缩的是行? 因为这样求出的平均值是列平均值, 而列平均值表示单个神经元所有参数的平均值
bn_meani = h_preact_batch.sum(0, keepdim=True) * (1 / batch_size)
# bn_meani 会 broadcast(自动扩展行)
# bn_diff 的均值是 0
bn_diff = h_preact_batch - bn_meani
bn_diff2 = bn_diff**2
bn_var = bn_diff2.sum(0, keepdim=True) * (
    1 / (batch_size - 1)
)  # 修改：除以 (n-1) 而不是 n（Bessel修正), 详细解释见下
bn_var_inv = (bn_var + 1e-5) ** (-0.5)
bn_raw = bn_diff * bn_var_inv  # 均值 0, 方差 1, 就是经过处理的 h_preact_batch
bn_hpreact = bn_gain * bn_raw + bn_bias
```

In [26]:
################# backward of batch normalization #########################
bn_raw.shape

torch.Size([32, 64])

$$
\begin{array}{l}
\hat{x_i} = \dfrac{x_i - \mu}{\delta} \\
\\
\small{\text{其中}} \\
\\
\mu = \dfrac{\left(\sum \limits_{i=0} ^{n=64} {x_i}\right)}{n} \\
\\
\delta = \dfrac{\left(x_i -  \sum \limits_{i=0} ^{n=64} {x_i}\right)}{n} \\
\\
x_i \small{\text{ 从三方面影响 }}\hat{x_i} \\
\\
x_i \small{\text{本身}} \\
\\
\dfrac{d(hat{x_i})}{d(x_i)} = 





\end{array}
$$